# Explore the trained model

Everything below runs against your **real, trained checkpoint** (`checkpoints/gpt_shakespeare.pt`) — real prompts in, real generated text out, real probabilities, not the hand-picked illustrative numbers from the Merge Ledger / Attention Loom / Sampling Lab artifacts.

Run cells top to bottom once, then go back and edit any cell — change the prompt, the temperature, the word you're checking neighbours for — and re-run just that cell to see the new output immediately below it. That's the whole point of a notebook over a script: change one thing, rerun, look, repeat.

In [1]:
import torch
import torch.nn.functional as F
from model.gpt import GPT
from model.sample import sample
from tokenizer.bpe import BPETokenizer

device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"device: {device}")

device: mps


In [2]:
tok = BPETokenizer()
tok.load("tokenizer/vocab.json")

ckpt = torch.load("checkpoints/gpt_shakespeare.pt", map_location=device)
model = GPT(**ckpt["config"]).to(device)
model.load_state_dict(ckpt["model"])
model.eval()

print(f"loaded checkpoint from iteration {ckpt['iter']}")
print(f"config: {ckpt['config']}")
print(f"params: {model.num_params():,}")

loaded checkpoint from iteration 6000
config: {'vocab_size': 1024, 'n_embd': 128, 'n_head': 4, 'n_layer': 4, 'block_size': 128, 'dropout': 0.1}
params: 922,880


## Generate text — try your own prompt

Change `prompt` to anything (it doesn't have to be a character name — try a random word or your own sentence start) and re-run.

In [3]:
@torch.no_grad()
def generate(prompt, max_new_tokens=80, temperature=0.8, top_k=40, top_p=None):
    ids = tok.encode(prompt)
    for _ in range(max_new_tokens):
        x = torch.tensor([ids[-model.block_size:]], device=device)
        logits = model(x)
        next_id = sample(logits[0, -1], temperature=temperature, top_k=top_k, top_p=top_p).item()
        ids.append(next_id)
    return tok.decode(ids)

print(generate("ROMEO:"))

ROMEO: but thou sort the heart of my fird!

MARGLOUCESTER:
What, is the tice?

ROMEO:
But my lord.

JALTOLIO:
Now, that gentlecome a gurew, in deswon
That dread and facection; on miss as fuped yours,
His I


## Temperature, on the same prompt

Same prompt, four temperatures. Compare against dragging the slider in the Sampling Lab artifact — same effect, real model this time.

In [4]:
prompt = "JULIET:"
for temp in [0.3, 0.8, 1.2, 1.8]:
    print(f"--- temperature={temp} ---")
    print(generate(prompt, temperature=temp, top_k=None))
    print()

--- temperature=0.3 ---
JULIET: I well, sir, he is not the crown,
Therefore the houch rever, stay this way,
Thanish'd to before the ways the words;
O have servings the curder, sint the blood
That turn night; but as the beath, that wh

--- temperature=0.8 ---
JULIET: he Marve forsonous east.

JINCII:
But wom Ceat the fection. Noper; make pice,
That he grant him, stin clos. On, for Romeoman's lifer
As yor sast to the kinder to him?

KING K:
Arings, thou

--- temperature=1.2 ---
JULIET:save Chause you can cect? be uda pertagwsack.

GORGK:
Fits denoon: he t's conted any most wears
Again to racewarmanulk the prase;
Lio afefore thy plaan, ch of of on soressince, and from

--- temperature=1.8 ---
JULIET: this chricks to press to arm: by do an un inthink wsir, my hath de.

DUKE Ed<QUEEN ES:
T: he daughes hace a not.

KING RICHARD III:

O hear mook are mespise were heares
me much they shaconce, feee.
Whilpo shawarg heat�MLirst y'st have nowale's death.

B



## The real next-token probabilities

The actual top-10 candidates and their real probabilities, straight out of the trained model — the same shape of thing the Sampling Lab showed you with made-up numbers, now with genuine ones.

In [5]:
prompt = "ROMEO:"
ids = tok.encode(prompt)
x = torch.tensor([ids[-model.block_size:]], device=device)
with torch.no_grad():
    logits = model(x)[0, -1]
probs = F.softmax(logits, dim=-1)
top_probs, top_ids = torch.topk(probs, 10)

print(f"after {prompt!r}, the model's real top-10 next tokens:")
for p, i in zip(top_probs.tolist(), top_ids.tolist()):
    print(f"  {p*100:5.1f}%  {tok.decode([i])!r}")

after 'ROMEO:', the model's real top-10 next tokens:
    7.5%  ' but '
    6.1%  ' I'
    5.1%  ' th'
    4.8%  ' the '
    4.6%  ' wh'
    4.3%  ' what '
    3.5%  ' m'
    3.3%  ' s'
    3.2%  ' he '
    2.8%  ' a'


## Embeddings, now that they're actually trained

Milestone 2's embeddings were **random noise** — nearest neighbours were meaningless because nothing had been learned yet. Now that the model is trained, check if similar words end up with similar vectors.

In [6]:
def nearest_neighbors(word, k=8):
    ids = tok.encode(word)
    print(f"{word!r} tokenizes to: {[tok.decode([i]) for i in ids]}")
    # pick the LONGEST sub-token as the representative one -- the
    # first token is often just a leading space, which every word shares
    target_id = max(ids, key=lambda i: len(tok.vocab[i]))
    all_vecs = model.token_emb.weight
    target_vec = all_vecs[target_id]
    sims = F.cosine_similarity(target_vec.unsqueeze(0), all_vecs, dim=-1)
    top_sims, top_ids = torch.topk(sims, k + 1)  # +1 because it always includes itself
    print(f"nearest neighbours of {tok.decode([target_id])!r}:")
    for s, i in zip(top_sims.tolist(), top_ids.tolist()):
        if i == target_id:
            continue
        print(f"  {s:.3f}  {tok.decode([i])!r}")

nearest_neighbors(" love")
print()
nearest_neighbors(" king")

' love' tokenizes to: [' ', 'lov', 'e']
nearest neighbours of 'lov':
  0.716  'liv'
  0.682  'do'
  0.671  'lif'
  0.651  'tru'
  0.644  'father'
  0.635  'rep'
  0.633  'queen'
  0.629  'tim'

' king' tokenizes to: [' ', 'king']
nearest neighbours of 'king':
  0.591  'queen'
  0.568  'father'
  0.544  'de'
  0.537  'noble '
  0.536  'death'
  0.535  'fa'
  0.528  'k'
  0.522  'son'


## Greedy vs. sampled

Greedy always picks the single most likely token — run it twice, get identical output. Sampling draws from the distribution — run it three times, watch it actually vary.

In [7]:
prompt = "KING:"

print("greedy (deterministic — run this cell again, it won't change):")
print(generate(prompt, temperature=0))
print()

print("sampled 3x at temperature=1.0 (watch it vary):")
for _ in range(3):
    print(generate(prompt, temperature=1.0, top_k=40))
    print()

greedy (deterministic — run this cell again, it won't change):
KING: but I do not not a pover, that he is a poy.
His the serving, that he is the soul
That breaty to the crown our blood:
But if you are not not not a povere: which are not the soul
As I do not a povere. 

sampled 3x at temperature=1.0 (watch it vary):
KING:, if you vowly brak?

SALET:
And we be haty with sast as the acher,
And with a laps from the heaven noing,
O love me it from all there to your been, my wid:
Te be greven ance of Gook
Bercark you

KING: what briefs have preven to thy mother.

GLOUCESTER:
He's my follows the gentleman to this underon, my trange it
A deolar of your goweak well.
My bopal wine natence prood, and chapery bush,
Shause with herself

KING: that no man:
If he was the firding houble hold try: for he shall are a king.
So sin them, I fus, when the soulth, then, let them ehiss
Which in you am as pition; yet doth this drice,
And beman cruy of us to trice.

L

